# 视频分类 X3D

In [1]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models

from torchvision.transforms import v2
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from pytorchvideo.models.hub import x3d_s

from concurrent.futures import ThreadPoolExecutor, as_completed

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## 一、处理数据

In [2]:
ucf_root_dir = r"/mnt/Data/daihongliang"
ucf_dataset_name = "UCF-101"
ucf_processed_name = "UCF-101-Processed"
ucf_train_test_file_dir = "ucfTrainTestlist"

In [3]:
def get_train_test_lists(ucf_root_dir, ucf_dataset_name, preprocess=False, tag=None, target=(256, 256), confirm=False, num_workers=8):
    data_dir = os.path.join(ucf_root_dir, ucf_dataset_name)
    class_list = os.listdir(data_dir)
    data_list = []
    preprocess_data_list = []

    for i in range(len(class_list)):
        class_item_dir = os.path.join(data_dir, class_list[i])
        data_list.extend([os.path.join(class_item_dir, it) for it in os.listdir(class_item_dir)])

    # === 多线程预处理部分 ===
    if preprocess:
        def process_video(video_path):
            try:
                cap = cv2.VideoCapture(video_path)
                array_save_path = video_path.replace(ucf_dataset_name, f'{ucf_dataset_name}-Processed')
                array_save_path = array_save_path.split('.')[0] + '.npy'
                if tag is not None:
                    array_save_path = array_save_path.split('.')[0] + f'-{tag}' + '.npy'
                preprocess_data_list.append(array_save_path)
                if os.path.exists(array_save_path):
                    cap.release()
                    return None

                os.makedirs(os.path.dirname(array_save_path), exist_ok=True)
                frames = []

                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame_resized = cv2.resize(frame_rgb, target)
                    frame_chw = np.transpose(frame_resized, (2, 0, 1))
                    frames.append(frame_chw)

                cap.release()
                np.save(array_save_path, np.stack(frames))
            except Exception as e:
                print(f"[Error] {video_path}: {e}")

        # 使用线程池并行处理
        if not confirm:
            with ThreadPoolExecutor(max_workers=num_workers) as executor:
                futures = [executor.submit(process_video, v) for v in data_list]
                for _ in tqdm(as_completed(futures), total=len(futures), desc="preprocess"):
                    pass
        else:
            for video_path in tqdm(data_list, desc='preprocess'):
                array_save_path = video_path.replace(ucf_dataset_name, f'{ucf_dataset_name}-Processed')
                array_save_path = array_save_path.split('.')[0] + '.npy'
                if tag is not None:
                    array_save_path = array_save_path.split('.')[0] + f'-{tag}' + '.npy'
                preprocess_data_list.append(array_save_path)


    # === 划分训练/测试集 ===
    if preprocess:
        train_list, test_list = train_test_split(preprocess_data_list, test_size=0.2)
    else:
        train_list, test_list = train_test_split(data_list, test_size=0.2)
    return train_list, test_list


In [4]:
train_data, test_data = get_train_test_lists(
    ucf_root_dir,
    ucf_dataset_name,
    preprocess=True,
    tag="192",
    target=(192, 192),
    confirm=True
)
print("Train: ", train_data[:2], "\nTest: ", test_data[:2])

preprocess: 100%|██████████| 13320/13320 [00:00<00:00, 1371333.56it/s]

Train:  ['/mnt/Data/daihongliang/UCF-101-Processed/PlayingDaf/v_PlayingDaf_g02_c07-192.npy', '/mnt/Data/daihongliang/UCF-101-Processed/BlowDryHair/v_BlowDryHair_g12_c06-192.npy'] 
Test:  ['/mnt/Data/daihongliang/UCF-101-Processed/PlayingSitar/v_PlayingSitar_g06_c02-192.npy', '/mnt/Data/daihongliang/UCF-101-Processed/Hammering/v_Hammering_g19_c05-192.npy']


In [5]:
classes = {}
classes_file = os.path.join(ucf_root_dir, ucf_train_test_file_dir, 'classInd.txt')
with open(classes_file) as fin:
    for row in fin:
        line_info = row.strip().split(' ')
        classes[line_info[1]] = int(line_info[0]) - 1
list(classes.items())[:2]

[('ApplyEyeMakeup', 0), ('ApplyLipstick', 1)]

## 二、编写 Torch 的 Dataset

In [ ]:
class UCFDataset(Dataset):
    def __init__(self, data, classes):
        self.num_frames = 64

        self.data = data
        self.classes = classes

        # 数据处理
        self.transform = v2.Compose(
            [
                v2.Resize(192),
                v2.CenterCrop(160),
                v2.ConvertImageDtype(torch.float32),
                v2.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
           ]
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        np_data_path = self.data[i]
        np_data = np.load(np_data_path)

        total_frames = np_data.shape[0]

        # 选择帧
        if total_frames <= self.num_frames:
            sample_data = np_data
        else:
            max_start = total_frames - self.num_frames
            start_frame = random.randint(0, max_start)
            sample_data = np_data[start_frame: start_frame + self.num_frames, :, :, :]

        frames = torch.tensor(sample_data).float()
        frames = self.transform(frames)

        label = self.classes[np_data_path.split('/')[-2]]
        return frames, label

In [15]:
class UCFDatasetV2(Dataset):
    def __init__(self, data, classes):
        self.num_frames = 13  # 改为13帧

        self.data = data
        self.classes = classes

        # 视频帧预处理
        self.transform = v2.Compose([
            v2.Resize(192),
            v2.CenterCrop(160),
            v2.ConvertImageDtype(torch.float32),
            v2.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        np_data_path = self.data[i]
        np_data = np.load(np_data_path)  # [T, H, W, C] or [T, C, H, W]

        total_frames = np_data.shape[0]

        # --- 修改部分：等间隔取13帧 ---
        if total_frames < self.num_frames:
            # 若视频帧不足13，重复最后一帧
            indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
            indices = np.clip(indices, 0, total_frames - 1)
        else:
            # 等间隔采样13帧
            indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)

        sample_data = np_data[indices, :, :, :]  # 取出这些帧
        # --------------------------------

        # 转成Tensor并调整维度
        frames = torch.tensor(sample_data)
        frames = self.transform(frames).permute(1, 0, 2, 3)

        label_name = np_data_path.split('/')[-2]
        label = self.classes[label_name]

        return frames, label

In [16]:
trainset = UCFDatasetV2(train_data, classes)
testset = UCFDatasetV2(test_data, classes)
d, l = trainset.__getitem__(123)
print(d.shape, l)

torch.Size([3, 13, 160, 160]) 91


## 三、编写 Torch 的 DataLoader

In [ ]:
def collate_fn(batch):
    data = []
    label = []

    # 筛选总帧数小于64的样本
    for d, l in batch:
        num_frames = d.shape[0]
        if num_frames < 64:
            continue  # 跳过帧数不足的样本

        indices = torch.linspace(0, num_frames - 1, 13).long()
        sampled_frames = d[indices]

        data.append(sampled_frames)
        label.append(l)

    # 处理批量中无有效样本的情况
    if len(data) == 0:
        return torch.tensor([]), torch.tensor([])

    # 组装张量（所有样本采样后帧数相同）
    data_tensor = torch.stack(data, dim=0).type(torch.FloatTensor).transpose(1, 2).contiguous()
    label_tensor = torch.tensor(label).type(torch.FloatTensor)

    return data_tensor, label_tensor

In [17]:
trainloader = DataLoader(trainset, batch_size=4, shuffle=True, num_workers=4)
testloader = DataLoader(testset, batch_size=4, shuffle=True, num_workers=4)
batch_data, batch_labels = next(iter(trainloader))
print("数据形状:", batch_data.shape)
print("标签形状:", batch_labels.shape)

数据形状: torch.Size([4, 3, 13, 160, 160])
标签形状: torch.Size([4])


## 四、设计网络

In [18]:
class X3D(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        self.model = x3d_s(pretrained=pretrained)

        self._change_num_classes(num_classes)

    def _change_num_classes(self, num_classes):
        # pytorchvideo 的 X3D 通常结构： model.blocks[-1].proj
        if hasattr(self.model, "blocks") and hasattr(self.model.blocks[-1], "proj"):
            in_features = self.model.blocks[-1].proj.in_features
            self.model.blocks[-1].proj = nn.Linear(in_features, num_classes)
        elif hasattr(self.model, "head") and hasattr(self.model.head, "proj"):
            in_features = self.model.head.proj.in_features
            self.model.head.proj = nn.Linear(in_features, num_classes)
        else:
            raise AttributeError("Unsupported X3D architecture; cannot find classification head")

    def forward(self, x):
        return self.model(x)

In [19]:
net = X3D(101)
o = net(batch_data)
print(o.shape)

Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/X3D_S.pyth" to /home/daihongliang@kemande.cn/.cache/torch/hub/checkpoints/X3D_S.pyth
100%|██████████| 29.4M/29.4M [00:02<00:00, 14.1MB/s]


torch.Size([4, 101])


## 五、构建训练代码

In [20]:
seed = 1
torch.manual_seed(seed)  # 为CPU设置随机种子
torch.cuda.manual_seed(seed)  # 为当前GPU设置随机种子
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU，为所有GPU设置随机种子
np.random.seed(seed)  # Numpy module.
random.seed(seed)  # Python random module.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [21]:
save_root = "./checkpoints/X3D"
num_classes = 101
learning_rate = 1e-4
train_epochs = 100
lr_scheduler_step = 50

In [22]:
# 创建检查目录
os.makedirs(os.path.join(save_root, 'acc'), exist_ok=True)
os.makedirs(os.path.join(save_root, 'loss'), exist_ok=True)

In [23]:
# model = X3D(101)
# device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
# model = model.to(device)
model = X3D(101)
if torch.cuda.device_count() > 1:
    print(f"使用 {torch.cuda.device_count()} 个GPU进行训练。")
    model = nn.DataParallel(model)  # 包装模型
model.cuda()

使用 4 个GPU进行训练。


DataParallel(
  (module): X3D(
    (model): Net(
      (blocks): ModuleList(
        (0): ResNetBasicStem(
          (conv): Conv2plus1d(
            (conv_t): Conv3d(3, 24, kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), bias=False)
            (conv_xy): Conv3d(24, 24, kernel_size=(5, 1, 1), stride=(1, 1, 1), padding=(2, 0, 0), groups=24, bias=False)
          )
          (norm): BatchNorm3d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation): ReLU()
        )
        (1): ResStage(
          (res_blocks): ModuleList(
            (0): ResBlock(
              (branch1_conv): Conv3d(24, 24, kernel_size=(1, 1, 1), stride=(1, 2, 2), bias=False)
              (branch2): BottleneckBlock(
                (conv_a): Conv3d(24, 54, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
                (norm_a): BatchNorm3d(54, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (act_a): ReLU()
                (conv_b)

In [24]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=lr_scheduler_step, gamma=0.1)
criterion = nn.CrossEntropyLoss()

In [25]:
def calculate_metrics(y_true, y_pred):
    """计算各种评估指标"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    return accuracy, precision, recall, micro_f1, macro_f1

In [ ]:
best_accuracy = 0
min_loss = float('inf')

for epoch in range(1, train_epochs + 1):
    print(f"\n{'=' * 60}")
    print(f"Epoch {epoch}/{train_epochs}")
    print(f"{'=' * 60}")

    # 更新学习率
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"当前学习率: {current_lr}")

    # 训练循环
    model.train()
    train_loss, train_correct, train_num = 0.0, 0, 0
    train_pred, train_label = [], []
    for _, (x, y) in enumerate(tqdm(trainloader, desc="训练中")):
        if not y.shape[0]:
            continue
        x, y = x.cuda(), y.long().cuda()

        # 梯度清零
        optimizer.zero_grad()

        # 前向传播
        outputs = model(x)
        loss = criterion(outputs, y)

        # 反向传播
        loss.backward()
        optimizer.step()

        # 统计信息
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_correct += (predicted == y).sum().item()
        train_num += y.size(0)

        train_pred.extend(predicted.cpu().numpy())
        train_label.extend(y.cpu().numpy())

    train_avg_loss = train_loss / len(trainloader)
    train_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(train_label, train_pred)
    print(
        f"Loss: {train_avg_loss:.4f}, Accuracy: {train_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

    # 测试循环
    model.eval()
    test_loss, test_correct_num, test_num = 0.0, 0, 0
    test_pred, test_label = [], []

    with torch.no_grad():
        for _, (x, y) in enumerate(tqdm(testloader, desc="测试中")):
            if not y.shape[0]:
                continue
            x, y = x.cuda(), y.long().cuda()

            outputs = model(x)
            loss = criterion(outputs, y)

            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_correct_num += (predicted == y).sum().item()
            test_num += y.size(0)

            test_pred.extend(predicted.cpu().numpy())
            test_label.extend(y.cpu().numpy())

    test_avg_loss = test_loss / len(testloader)
    test_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(test_label, test_pred)
    print(
        f"Loss: {test_avg_loss:.4f}, Accuracy: {test_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

    # 保存最佳准确率模型
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        save_path = os.path.join(save_root, 'acc', f'best_acc.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")

    # 保存最低损失模型
    if test_avg_loss < min_loss:
        min_loss = test_avg_loss
        save_path = os.path.join(save_root, 'loss', f'best_loss.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")

    # 正确率超过99%时结束训练
    if test_accuracy >= 0.99:
        print(f"训练准确率已达到 {test_accuracy:.2%}，提前终止训练")
        save_path = os.path.join(save_root, f'early_stop_epoch_{epoch}.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
        break



/home/daihongliang@kemande.cn/miniconda3/envs/pytorch/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(



Epoch 1/100
当前学习率: 0.0001


训练中: 100%|██████████| 2664/2664 [08:07<00:00,  5.46it/s]


Loss: 4.4222, Accuracy: 24.01%, Precision: 0.1626, Recall: 0.2401, Micro F1: 0.2401, Macro F1: 0.1528


测试中: 100%|██████████| 666/666 [01:53<00:00,  5.89it/s]


...

训练中: 100%|██████████| 2664/2664 [08:25<00:00,  5.27it/s]


Loss: 4.1090, Accuracy: 56.80%, Precision: 0.4599, Recall: 0.5680, Micro F1: 0.5680, Macro F1: 0.4666


测试中: 100%|██████████| 666/666 [01:52<00:00,  5.92it/s]


Loss: 4.0590, Accuracy: 63.66%, Precision: 0.5444, Recall: 0.6366, Micro F1: 0.6366, Macro F1: 0.5419
模型已保存: ./checkpoints/X3D/acc/best_acc.pth (Epoch 2, Acc: 63.66%, Loss: 4.0590)
模型已保存: ./checkpoints/X3D/loss/best_loss.pth (Epoch 2, Acc: 63.66%, Loss: 4.0590)

Epoch 3/100
当前学习率: 0.0001


训练中:  29%|██▉       | 776/2664 [02:32<06:06,  5.15it/s]